
# Phase 3 — Model Development

This notebook evaluates **four machine learning algorithms**:

1. Logistic Regression (Baseline)
2. Decision Tree
3. Random Forest
4. Gradient Boosting

Workflow:

• Feature selection including **age, gender, department**  
• **Leakage-safe time-based train/test split**  
• **ColumnTransformer** preprocessing for categorical variables  
• **Pipeline integration**  
• Evaluation of all models  
• **Model comparison table**  
• **GridSearchCV hyperparameter tuning**  
• Saving the best model artifact


In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import joblib


## Load Dataset

In [20]:
df = pd.read_csv("../phase2/model_table.csv")
df.head()


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approved_amount,claim_status,payment_days,billing_date,visit_frequency,avg_los_per_patient,provider_rejection_rate,days_since_registration,visit_month,visit_dayofweek
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.00,Rejected,16.0,2025-06-18,2,3.725000,0.148655,65,10,5
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,38178.81,Paid,18.0,2025-10-09,4,32.025000,0.156915,-206,4,6
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,5038.97,Paid,NaN,2025-01-20,4,20.542500,0.149678,9,7,6
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,22813.34,Paid,16.0,2025-12-24,7,28.165714,0.152480,-62,11,2
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,27106.95,Paid,14.0,2025-09-23,5,22.988000,0.149678,0,3,5


## Feature Selection

In [23]:
target = "risk_score"

features = [
    "age",
    "gender",
    "department",
    "visit_type",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "days_since_registration",
    "visit_month",
    "visit_dayofweek"]


## Time-based Train/Test Split

In [24]:
df = df.sort_values("visit_date")

split_index = int(len(df)*0.8)

train = df.iloc[:split_index]
test = df.iloc[split_index:]

X_train = train[features].fillna(0)
X_test = test[features].fillna(0)

y_train = train[target]
y_test = test[target]


## ColumnTransformer Preprocessing

In [25]:
categorical_features = ["gender","department","visit_type"]

numerical_features = [
    "age","length_of_stay_hours","visit_frequency",
    "avg_los_per_patient","provider_rejection_rate","days_since_registration"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_features),
        ("num","passthrough",numerical_features)
    ]
)


## Define Models

In [26]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier()
}

results = []
best_model = None
best_score = 0


## Train and Evaluate Models

In [27]:
for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)

    acc = accuracy_score(y_test, preds)

    print("\nModel:", name)
    print("Accuracy:", acc)
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))

    results.append((name, acc))

    if acc > best_score:
        best_score = acc
        best_model = pipeline



Model: Logistic Regression
Accuracy: 0.4968
[[   0 1019    0]
 [   0 2484    0]
 [   0 1497    0]]
              precision    recall  f1-score   support

        High       0.00      0.00      0.00      1019
         Low       0.50      1.00      0.66      2484
      Medium       0.00      0.00      0.00      1497

    accuracy                           0.50      5000
   macro avg       0.17      0.33      0.22      5000
weighted avg       0.25      0.50      0.33      5000


Model: Decision Tree
Accuracy: 0.3706
[[ 220  444  355]
 [ 498 1116  870]
 [ 283  697  517]]
              precision    recall  f1-score   support

        High       0.22      0.22      0.22      1019
         Low       0.49      0.45      0.47      2484
      Medium       0.30      0.35      0.32      1497

    accuracy                           0.37      5000
   macro avg       0.34      0.34      0.34      5000
weighted avg       0.38      0.37      0.37      5000


Model: Random Forest
Accuracy: 0.4656
[[  3

## Model Comparison Table

In [28]:
comparison = pd.DataFrame(results, columns=["Model","Accuracy"])
comparison.sort_values("Accuracy", ascending=False)


,Model,Accuracy
0,Logistic Regression,0.4968
3,Gradient Boosting,0.4682
2,Random Forest,0.4656
1,Decision Tree,0.3706


## Hyperparameter Tuning (Random Forest Example)

In [29]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}



rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

grid = GridSearchCV(rf_pipeline,param_grid,cv=3,scoring="f1_macro",n_jobs=-1)

grid.fit(X_train,y_train)

print("Best Parameters:",grid.best_params_)

best_model = grid.best_estimator_


Best Parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}


## Final Evaluation

In [30]:
preds = best_model.predict(X_test)

print("Best ModeL:",best_model)
print("Final Accuracy:", accuracy_score(y_test,preds))
print(confusion_matrix(y_test,preds))
print(classification_report(y_test,preds))


Best ModeL: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'department',
                                                   'visit_type']),
                                                 ('num', 'passthrough',
                                                  ['age',
                                                   'length_of_stay_hours',
                                                   'visit_frequency',
                                                   'avg_los_per_patient',
                                                   'provider_rejection_rate',
                                                   'days_since_registration'])])),
                ('model', RandomForestClassifier(random_state=42))])
Final Accuracy: 0.4656
[[  30  805  184]
 [  62 2034  388]
 [  59 1174  264]]
         

## Save Model Artifact

In [31]:
joblib.dump(best_model,r"F:\AI ML\capstone\phase5\hospital_prediction_system\models\risk_model.pkl",compress=3)
print("risk_model.pkl saved")


risk_model.pkl saved


In [32]:
import json
from sklearn.preprocessing import OneHotEncoder

pipeline = best_model
preprocessor = pipeline.named_steps["preprocessor"]

feature_schema = {}

for name, transformer, columns in preprocessor.transformers_:

    # Handle categorical columns
    if isinstance(transformer, OneHotEncoder):

        categories = transformer.categories_

        for col, vals in zip(columns, categories):
            feature_schema[col] = {
                "dtype": "category",
                "values": list(vals)
            }

    # Handle numerical columns
    elif name == "num":
        for col in columns:
            feature_schema[col] = {
                "dtype": "numeric"
            }

schema = {
    "model_name": "hospital_prediction_model",
    "features": feature_schema
}

with open("risk_features.json", "w") as f:
    json.dump(schema, f, indent=4)

print("risk_features.json created successfully")

risk_features.json created successfully
